# GeoAgent Quickstart

This notebook walks through the core GeoAgent workflow:
1. Load a project
2. Explore wells and seismic surveys
3. Plot a well log panel
4. Generate a location map
5. Generate a synthetic seismogram

In [ ]:
# Install if needed:
# !pip install -e /path/to/geoagent

import geoagent
print(f'GeoAgent v{geoagent.__version__}')

## 1. Load a Project

Point `CoreDataManager` at your project directory containing `.pkl` data files.

In [ ]:
from geoagent import CoreDataManager

# Replace with your project path
PROJECT_DIR = r'C:\Delete_ML_Projects\test2'

dm = CoreDataManager(PROJECT_DIR)
print(dm)

## 2. Explore Available Data

In [ ]:
# List wells
wells = dm.get_available_wells()
print(f'Wells: {len(wells)}')

# List surveys and attributes
for survey in dm.get_available_surveys():
    attrs = dm.seismic_handler.get_available_attributes(survey)
    print(f'Survey: {survey} — {len(attrs)} attributes')

In [ ]:
# Inspect well heads
well_heads = dm.get_data('well_heads')
well_heads.head()

## 3. Plot a Well Log Panel

In [ ]:
%matplotlib inline
from geoagent.plotting.well_panel import plot_well_panel

# Pick a well that has logs
well_name = 'BK-14'  # Change to a well in your project

logs_df = dm.well_log_handler.get_well_logs(well_name)
if logs_df is not None:
    depth_col = next(c for c in logs_df.columns if c.upper() in ['DEPT', 'DEPTH', 'MD'])
    depth = logs_df[depth_col].values
    logs = {col: logs_df[col].values for col in logs_df.columns if col != depth_col}
    
    fig, axes = plot_well_panel(depth, logs, well_name=well_name)
else:
    print(f'No logs for {well_name}')

## 4. Generate a Synthetic Seismogram

This requires upscaled logs and a TDR (time-depth relationship).

In [ ]:
import numpy as np
from geoagent.synthetic.functions import create_reflectivity, create_synthetic_seismic_valid

# Quick demo with synthetic impedance data
impedance = np.array([5000, 6000, 5500, 7000, 6500, 5800, 6200, 7500, 6800])
reflectivity = create_reflectivity(impedance)
print(f'Impedance: {len(impedance)} samples → Reflectivity: {len(reflectivity)} samples')

# Convolve with a Ricker wavelet
from scipy.signal import ricker
wavelet = ricker(51, 4)  # ~30 Hz
synthetic = create_synthetic_seismic_valid(reflectivity, wavelet)
print(f'Synthetic trace: {len(synthetic)} samples')

## Next Steps

- See `02_well_log_panel.py` for full well log display
- See `04_synthetic_tie.py` for a complete synthetic tie workflow
- See `05_bulk_shift_scan.py` for CC-scan bulk shift optimization
- See the `well_correlation.ipynb` notebook for cross-section panels